In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI ,GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [ ]:
loader = PyPDFLoader("../data/YT_Script_Audio_PRO.pdf")
docs = loader.load()

len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)

len(splitted_data)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [ ]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [ ]:
query = "how to improve audio quality"
data = vector_store.similarity_search(query=query)

In [ ]:
context = ""
for doc in data:
    context += doc.page_content + "\n"


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [ ]:
res = llm.invoke(f"""can you provide me the answer based on provided \
 context for my question , context : {context} and qestion :{query}""")

In [ ]:
print(res.content)

### Chain - Context_generate | Promt | LLM | strparser

In [ ]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [ ]:
prompt = PromptTemplate.from_template("""
    you are a helpful assistatn and provide answerd based on the context for user quetions.
    and if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Qestion: {question}
    """)

In [ ]:
rag_chain = get_context | prompt | llm

In [ ]:
res = rag_chain.invoke("how to improve audio")

In [ ]:
print(res)